# Load Data

In [11]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb
# Basic Models with just phase interaction
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'isBounty',
    'userFEIsBounty',
    'userId',
    'timeSinceFirstActivityDays',
    'logtimeSinceFirstActivityDays',
    'userFeLogTimeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'userFeLogNumHelpProvidedAT',
    'userFeLogNumQuestionsAskedAT',
]

df = pd.read_parquet('../data/study_datasets/user_answers_bounty_processed.parquet',
                    columns=required_columns)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32856258 entries, 0 to 32856257
Data columns (total 12 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   isBounty                             int64  
 1   userFEIsBounty                       float32
 2   userId                               int64  
 3   timeSinceFirstActivityDays           float64
 4   logtimeSinceFirstActivityDays        float64
 5   userFeLogTimeSinceFirstActivityDays  float32
 6   numHelpProvidedAT                    int64  
 7   numQuestionsAskedAT                  int64  
 8   logNumHelpProvidedAT                 float64
 9   logNumQuestionsAskedAT               float64
 10  userFeLogNumHelpProvidedAT           float32
 11  userFeLogNumQuestionsAskedAT         float32
dtypes: float32(4), float64(4), int64(4)
memory usage: 2.4 GB


# Get Descriptives

In [14]:
# Define columns for descriptive statistics
descriptive_columns = [
    'isBounty',
    'timeSinceFirstActivityDays',
    'numHelpProvidedAT',
    'numQuestionsAskedAT',
    'logNumHelpProvidedAT',
    'logNumQuestionsAskedAT',
    'logtimeSinceFirstActivityDays'
]

def calculate_descriptives(data, columns):
    """Calculate descriptive statistics (μ, σ, min, max) for specified columns"""
    stats_dict = {}

    for col in columns:
        if col in data.columns:
            # Handle non-numeric columns by skipping them or converting
            if data[col].dtype in ['object', 'category']:
                continue

            stats_dict[col] = {
                'μ': data[col].mean(),
                'σ': data[col].std(),
                'min': data[col].min(),
                'max': data[col].max()
            }
        else:
            print(f"Warning: Column '{col}' not found in data")
    return pd.DataFrame(stats_dict).T

# Calculate overall descriptive statistics
print("=== DESCRIPTIVE STATISTICS ===")
overall_stats = calculate_descriptives(df, descriptive_columns)
print(overall_stats.round(4))

=== DESCRIPTIVE STATISTICS ===
                                      μ          σ  min         max
isBounty                         0.0105     0.1021  0.0      1.0000
timeSinceFirstActivityDays     939.3076  1031.1634  0.0   6082.1702
numHelpProvidedAT              726.6533  3253.3316  0.0  85078.0000
numQuestionsAskedAT             13.9251    38.4722  0.0   3072.0000
logNumHelpProvidedAT             3.9837     2.3994  0.0     11.3513
logNumQuestionsAskedAT           1.5340     1.4117  0.0      8.0304
logtimeSinceFirstActivityDays    5.6642     2.2544  0.0      8.7133


# 1. Main Bounty Effect

In [15]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
import gc
import os

sys.modules['stargazer.translators.statsmodels'].pd = pd

# Non-FE Models
formula1 = "isBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logtimeSinceFirstActivityDays"

# FE Models
formula2 = "userFEIsBounty ~ userFeLogNumHelpProvidedAT + userFeLogNumQuestionsAskedAT + userFeLogTimeSinceFirstActivityDays"

model_names = ["1", "2"]
formulas = [formula1, formula2]

def fit_model(formula, model_name, data):
    """Fit a single model, save LaTeX results, and clean memory"""
    print(f"Fitting Model {model_name}...")
    print(f"Formula: {formula}")

    try:
        # Clear memory before fitting
        gc.collect()

        # Fit the model
        model = smf.ols(formula=formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['userId']}
        )

        print(f"✓ Model {model_name} fitted successfully")

        # Create individual Stargazer table for this model
        stargazer = Stargazer([model])
        stargazer.title(f"Model {model_name}: Effect of Receiving Answers on Providing Help")
        stargazer.significant_digits(3)
        stargazer.show_degrees_of_freedom(False)
        stargazer.show_model_numbers(True)

        # Generate LaTeX code
        latex_output = stargazer.render_latex()
        print(latex_output)

        # Clear the model from memory
        del model
        gc.collect()

        return True

    except MemoryError as e:
        print(f"✗ Memory error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False
    except Exception as e:
        print(f"✗ Error fitting model {model_name}: {str(e)}")
        gc.collect()
        return False

successful_models = []
failed_models = []

for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"Processing {i+1}/{len(formulas)}")
    fit_model(formula, name, df)

Processing 1/2
Fitting Model 1...
Formula: isBounty ~ logNumQuestionsAskedAT + logNumHelpProvidedAT + logtimeSinceFirstActivityDays
✓ Model 1 fitted successfully
\begin{table}[!htbp] \centering
  \caption{Model 1: Effect of Receiving Answers on Providing Help}
\begin{tabular}{@{\extracolsep{5pt}}lc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{1}{c}{\textit{Dependent variable: isBounty}} \
\cr \cline{2-2}
\\[-1.8ex] & (1) \\
\hline \\[-1.8ex]
 Intercept & 0.010$^{***}$ \\
& (0.000) \\
 logNumHelpProvidedAT & -0.001$^{***}$ \\
& (0.000) \\
 logNumQuestionsAskedAT & -0.001$^{***}$ \\
& (0.000) \\
 logtimeSinceFirstActivityDays & 0.001$^{***}$ \\
& (0.000) \\
\hline \\[-1.8ex]
 Observations & 32856258 \\
 $R^2$ & 0.001 \\
 Adjusted $R^2$ & 0.001 \\
 Residual Std. Error & 0.102 \\
 F Statistic & 721.248$^{***}$ \\
\hline
\hline \\[-1.8ex]
\textit{Note:} & \multicolumn{1}{r}{$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01} \\
\end{tabular}
\end{table}
Processing 2/2
Fitting Model 2...